# Machine Learning Analysis: Credit Risk & Loan Default Classification
### Model Training Pipeline: Logistic Regression

---

## Project Overview
This notebook implements a end-to-end machine learning pipeline to classify and predict loan defaults using a credit performance dataset from Kaggle. 

Because we engineered highly informative **stratified probability** and **Weight of Evidence (WoE)** features, our categorical data now shares a linear relationship with the log-odds of the target variable. This makes **Logistic Regression** the mathematically ideal model for this analysis. 

### Pipeline Architecture:
1. **Data Preparation:** Splitting the dataset into stratified training and testing sets to prevent data leakage.
2. **Feature Scaling:** Standardizing continuous variables to ensure stable gradient descent convergence.
3. **Model Fitting:** Training a penalized Logistic Regression model.
4. **Evaluation:** Assessing performance using class-imbalance-resilient metrics including ROC-AUC, Precision-Recall curves, and F1-Score.

## 1. Initial Data Loading & Environmental Setup
In this initial step, we load the raw loan performance dataset into a Pandas DataFrame. 

In [5]:
import kagglehub
import pandas
import glob
import os

# Download latest version
path = kagglehub.dataset_download("nikhil1e9/loan-default")
# Joins the folder path with the wildcard pattern
csv_files = glob.glob(os.path.join(path, "*.csv"))

print("Path to dataset files:", path)
print(csv_files)

df = pandas.read_csv(csv_files[0])
categorical_columns = (df.select_dtypes(include=['str']) if pandas.__version__[0] >= '3' else df.select_dtypes(include=['object'])).columns[1:].tolist()
print(f'Categorical columns: {categorical_columns}')
df

Path to dataset files: C:\Users\victo\.cache\kagglehub\datasets\nikhil1e9\loan-default\versions\2
['C:\\Users\\victo\\.cache\\kagglehub\\datasets\\nikhil1e9\\loan-default\\versions\\2\\Loan_default.csv']
Categorical columns: ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255342,8C6S86ESGC,19,37979,210682,541,109,4,14.11,12,0.85,Bachelor's,Full-time,Married,No,No,Other,No,0
255343,98R4KDHNND,32,51953,189899,511,14,2,11.55,24,0.21,High School,Part-time,Divorced,No,No,Home,No,1
255344,XQK1UUUNGP,56,84820,208294,597,70,3,5.29,60,0.50,High School,Self-employed,Married,Yes,Yes,Auto,Yes,0
255345,JAO28CPL4H,42,85109,60575,809,40,1,20.90,48,0.44,High School,Part-time,Single,Yes,Yes,Other,No,0


## 2. Data Splitting: Train, Validation, and Test Sets
In this step, we partition our dataset into three distinct subsets—**Training (70%)**, **Validation (10%)**, and **Test (20%)**—using a **stratified split**. 

* **Training Set:** Used exclusively to fit our Logistic Regression model and calculate the stratified WoE/probability mappings.
* **Validation Set:** Used to tune hyperparameters (such as the regularization strength parameter, $C$) and prevent overfitting.
* **Test Set:** Held back as a final "blind" dataset to evaluate how well our model generalizes to completely unseen credit applications.

*Note: We apply **stratification** based on our target variable to ensure that the 11% default rate is precisely preserved across all three data subsets.*


In [2]:
from sklearn.model_selection import train_test_split


df['stratify_col'] = df[categorical_columns].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)

train_val_df_raw, test_df_raw = train_test_split(
    df, 
    test_size=0.20, 
    random_state=100, 
    stratify=df['stratify_col'] + '_' + df['Default'].astype(str) # <--- Stratifying on combined column
)

train_df_raw, val_df_raw = train_test_split(
    train_val_df_raw, 
    test_size=0.20, 
    random_state=100, 
    stratify=train_val_df_raw['stratify_col'] + '_' + train_val_df_raw['Default'].astype(str) # <--- Stratifying on combined column
)
del train_val_df_raw

print(f'Checking stratification: {train_df_raw["stratify_col"].unique().shape[0]} unique values in train set, {test_df_raw["stratify_col"].unique().shape[0]} unique values in test set')
print(f'Train size: {train_df_raw.shape[0]}, Validation size: {val_df_raw.shape[0]}, Test size: {test_df_raw.shape[0]}')

Checking stratification: 1920 unique values in train set, 1920 unique values in test set
Train size: 163421, Validation size: 40856, Test size: 51070


In [3]:
train_df = train_df_raw.copy()
val_df = val_df_raw.copy()
test_df = test_df_raw.copy()

## 3. Feature Engineering: Categorical-to-Numerical Transformation
In this section, we convert our **4 categorical variables** into highly predictive numerical features optimized for our Logistic Regression pipeline. This transformation ensures our models can interpret the categorical data while preserving complex non-linear relationships.

### Strategic Roadmap:
* **Stratified Combination:** Cross-tabulating individual categorical fields to create unique borrower strata.
* **Probability Mapping:** Converting the categorical strata into empirical probability-of-default features.
* **Weight of Evidence (WoE) Encoding:** Translating the categorical groups into numerical linear logs-odds scores to match the mathematical requirements of Logistic Regression.

## 3.1 Generating Weight of Evidence (WoE) Embedding Dictionaries
To apply our feature engineering strategy, we calculate the **Weight of Evidence (WoE)** values for our stratified categorical segments using the training partition. 

Instead of adding multiple structural columns to our dataframe right away, we store these calculated mappings inside a native **Python dictionary**. 

### Benefits of the Dictionary Mapping Approach:
* **Prevents Data Leakage:** Mappings are generated strictly from training data distributions and stored safely inside a reference dictionary object.
* **Lightweight & Fast:** The dictionary acts as a lookup table. We can instantly embed WoE values into the Train, Validation, and Test sets using a highly efficient `.map()` operation.
* **Production-Ready:** This dictionary format is highly portable and can easily be exported as a JSON file, making it simple to deploy into a live production scoring pipeline.


In [4]:
from utils import calculate_woe, woe_multiple
from utils import category_to_woe

woe_dict, iv_df = woe_multiple(train_df, ['stratify_col'], target='Default')

Total Information Value (IV) for 'stratify_col': 0.1358
Information Value Ranking:
                    IV
stratify_col  0.135782


## 3.2 Target Probability Encoding
In this step, we implement **Target Probability Encoding** on our stratified categorical fields. This technique replaces each categorical combination with its corresponding empirical probability of default ($P(\text{Default} | \text{Category})$) derived directly from the training data.

#### Implementation & Safeguards:
* **Linearizing Relationships:** This converts raw text categories into a continuous numerical range between $0$ and $1$, mapping a direct mathematical relationship to our target variable.
* **Leakage Prevention:** To avoid overfitting and data leakage, these probabilities are calculated strictly using the training partition. The resulting mapping matrix is then applied blindly as a lookup table to the validation and test datasets.

In [5]:
from statsmodels.stats.proportion import proportions_ztest

default_rate_by_all_categories = train_df.groupby('stratify_col')['Default'].agg(['mean', 'count']).sort_values('mean', ascending=False)
default_rate_by_all_categories['pval'] = default_rate_by_all_categories.apply(lambda row: proportions_ztest(count=[row['count'] * row['mean'],
                                                                                                                   train_df['Default'].mean() * train_df['Default'].count()],
                                                                                                                   nobs=[row['count'], train_df['Default'].count()],
                                                                                                                   alternative='two-sided')[1], axis=1)
default_rate_by_all_categories = default_rate_by_all_categories.rename(columns={'index': 'category_value', 'mean': 'default_rate', 'count': 'num_samples'})
default_rate_by_all_categories['significant'] = default_rate_by_all_categories['pval'] < 0.05
prob_dict = default_rate_by_all_categories[ 'default_rate'].to_dict()

### 3.3 Modular Pipeline: The `feature_engineering` Function
To ensure clean, reproducible, and production-ready code, we encapsulate our entire transformation logic into a single Python function. 

This modular function takes a raw dataset and our pre-calculated dictionary mappings (probabilities and WoE embeddings) as inputs, then applies the stratified transformations uniformly across the columns.

#### Key Engineering Benefits:
* **Consistency:** Applies identical mapping logic across Train, Validation, and Test sets, guaranteeing feature parity.
* **Leakage Prevention:** By feeding pre-computed dictionaries into the function, we ensure that validation and test data are never used during the initial encoding calculations.
* **Scalability:** Streamlines the deployment process, allowing the exact same data pipeline to process new, real-time loan applications in a production environment.


In [6]:
def feature_engineering(df, woe_dict, prob_dict):
    df = category_to_woe(df, ['stratify_col'], woe_dict)
    df['stratify_col_prob'] = df['stratify_col'].map(prob_dict)
    df = df.drop(columns=['LoanID', 'stratify_col'] + categorical_columns)
    return df

train_df = feature_engineering(train_df, woe_dict, prob_dict)
val_df = feature_engineering(val_df, woe_dict, prob_dict)
test_df = feature_engineering(test_df, woe_dict, prob_dict)

X_train = train_df.drop(columns=['Default'])
y_train = train_df['Default']
X_val = val_df.drop(columns=['Default'])
y_val = val_df['Default']
X_test = test_df.drop(columns=['Default'])
y_test = test_df['Default']

## 4. Machine Learning Experimentation & MLflow Tracking
In this final phase, we execute our Logistic Regression training pipeline within a formalized experiment tracking framework using **MLflow**. This ensures every iteration, metric, and artifact is fully reproducible and auditable.

Instead of writing verbose logging boilerplate, we utilize a standardized `MLflowLogger` wrapper imported from our local `utils` module to streamline the tracking workflow.

### Logging Protocol & Artifacts:
To capture a comprehensive performance profile of our Logistic Regression model, the logging sequence automatically records:
* **Performance Metrics:** Class-imbalance-resilient evaluation metrics including **ROC-AUC**, **Precision**, **Recall**, and the **F1-Score** computed across both validation and test partitions.
* **Visual Artifacts:** Saving generated diagnostic plots directly to the MLflow run artifact store (including the **Confusion Matrix**, **ROC Curve**, and **Precision-Recall Curve**).
* **Feature Importance:** Extracting and logging the model's standardized coefficients (weights) to clearly document the predictive impact of our engineered stratified features.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from utils import MLflowLogger

drop_cols = []

# Define which columns to scale and which to exclude
scale_cols = X_train.drop(columns=drop_cols).columns.difference([])  # Assuming 'stratify_col_prob' is the only column we want to exclude from scaling


# Your pipeline
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), scale_cols)
], remainder='passthrough')

params = {
}

logistic_model = LogisticRegression(**params)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', logistic_model)
])

logger = MLflowLogger(experiment_name="Credit_Risk_PD_Modeling")

final_model = logger.fit(
    model=pipeline,           # or raw model
    X_train=X_train.drop(columns=drop_cols),
    y_train=y_train,
    X_val=X_val.drop(columns=drop_cols),
    y_val=y_val,
    X_test=X_test.drop(columns=drop_cols),            # ← Optional but recommended
    y_test=y_test,
    run_name="LogisticRegression_with_prob_and_woe_scaled",
    model_name="LogisticRegression",
    description="Logistic Regression model with stratify_col_prob and stratify_col_woe scaled.",
    # fit parameters for LogisticRegression...
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=60,
    eval_metric='auc'
)

Training LogisticRegression...
   Train: AUC=0.7579 | Acc=0.8861 | F1=0.0920 | Prec=0.6204 | Rec=0.0497
   Val: AUC=0.7581 | Acc=0.8855 | F1=0.0889 | Prec=0.5938 | Rec=0.0480


2026/05/27 14:30:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


   Test: AUC=0.7506 | Acc=0.8858 | F1=0.0885 | Prec=0.5945 | Rec=0.0478
✅ Feature importance logged for train set


2026/05/27 14:30:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Run completed → LogisticRegression_with_prob_and_woe_scaled



## 5.1 Accessing the MLflow Dashboard
Once the training and logging pipeline completes, you can visualize the tracked metrics, parameters, and generated diagnostic plots through the local MLflow user interface. 

To launch the dashboard, execute the following bash command directly within a notebook cell:

```bash
!python -m mlflow ui --host 127.0.0.1 --port 5000
```

After running the command, navigate to your web browser and open the corresponding URL:
👉 **[http://127.0.0.1:5000](http://127.0.0.1:5000)**

*Note: Keep the cell running to maintain the server connection while reviewing your experiments.*